<b><font size="6" color="#E8800A">Week 3 · Deepening exploration: quality, missing values and outliers</font></b><br>
<b><font size="4">The one week whose output every later week runs on</font></b><br>

`champions_raw.csv` is the file as it arrives: **4,280 rows**, a
misspelt boolean, two columns carrying values their own units forbid, missing values in 26
of its 28 feature columns, and one athlete appearing in several competitions.
`champions.csv` is what this notebook produces from it: **4,000 rows**, the exact
frame every modelling week works on.

Every cleaning choice made here is therefore inherited by every score in the rest
of the course, which is why none of them is settled by convention. A convention is
what most people do; it is not evidence about this file. So each choice is written
as a **function that takes the alternatives as arguments**, the alternatives are
scored on identical held-out rows, and the recipe at the end is those same
functions called with whichever option won.

The target here is a class: `Outcome`, 1 if the athlete won. Every technique below
is chosen by the variable types in front of it, so the same method carries to a
numeric target without changing shape. `week_03_deepen_exploration_regression`
is the alternative that does exactly that, on a file of used car prices.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>The file, and the file it becomes</font>](#frame)
* [<font color='#E8800A'>Structural survey</font>](#survey)
* [<font color='#E8800A'>Distribution statistics</font>](#distributions)
* [<font color='#E8800A'>An invalid value is not always missing data</font>](#invalid)
* [<font color='#E8800A'>Where the boxplot fails</font>](#boxplot)
* [<font color='#E8800A'>The duplicates `drop_duplicates` misses</font>](#duplicates)
* [<font color='#E8800A'>From exploration to a recipe</font>](#plan)
* [<font color='#E8800A'>Impossible values: blank, clip or drop</font>](#invalidfix)
* [<font color='#E8800A'>Repeated athletes: which row to keep</font>](#tiebreak)
* [<font color='#E8800A'>Missing values: compare nine, then select</font>](#imputation)
* [<font color='#E8800A'>Outliers: compare five, then judge</font>](#outliers)
* [<font color='#E8800A'>Does scaling change anything here?</font>](#scaling)
* [<font color='#E8800A'>The recipe, the order, and the log</font>](#recipe)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>The file, and the file it becomes</font> <a class="anchor" id="frame"></a>
[Back to TOC](#toc)

The pipeline you have already run end to end used data that had been
tidied for you. This week is the tidying.

Two files sit in `data/raw/`. `champions_raw.csv` is what the
federation sent; `champions.csv` is what a documented recipe turns it into. Every
later week opens the second one and never sees the first.

That gives this notebook a property no other week has: **it can be checked
exactly.** A cleaning decision is usually defended with an argument. Here, the
final exercise runs your recipe and compares the result with `champions.csv` - this file's intended output.

Every import this notebook needs is in the single cell below, and nothing after
it imports anything. That is PEP 8 (*"imports are always put at the top of the
file"*), and it is machine-checkable: `pycodestyle` reports a late import as
**E402**.

**What is in the file.** One row per competition entry: `RecordID`
identifies the row and goes in the index, leaving 28 columns.
The target is **`Outcome`**: 1 if the athlete won that entry, 0 if not.

| column | what it holds | role |
|---|---|---|
| `Outcome` | 1 = won, 0 = did not | **target** |
| `RecordID` | one value per row | identifier, never a feature |
| `Athlete Id` | one value per athlete; an athlete can appear more than once | identifier, never a feature |
| `Competition` | which event: local match up to world championship (7 levels) | categorical |
| `Edition` | which running of the event (4 values) | categorical |
| `Sex` | `M` / `F` | categorical |
| `Region` | where the athlete competes from (13 levels) | categorical |
| `Education` | highest level completed (5 levels) | **ordinal** |
| `Age group` | banded age (4 levels) | **ordinal**, and one level is not a band |
| `Income` | banded income (5 levels) | **ordinal** |
| `Previous attempts` | how many times the athlete has entered before | numeric count |
| `Athlete score` | the federation's own rating of the athlete | numeric |
| `Disability` | registered disability | boolean |
| `Late enrollment` | entered after the deadline | boolean |
| `Mental preparation` | followed the mental-preparation programme | boolean |
| `Outdoor Workout` | trains outdoors | boolean |
| `No coach` | competes without a coach | boolean |
| `Past injuries` | has an injury history | boolean |
| `Train bf competition` | minutes, in the run-up to the event | numeric, training volume |
| `Strength training` | minutes | numeric, training volume |
| `Sand training` | minutes | numeric, training volume |
| `Recovery` | minutes | numeric, training volume |
| `Supplements` | minutes | numeric, training volume |
| `Cardiovascular training` | minutes | numeric, training volume |
| `Squad training` | minutes | numeric, training volume |
| `Physiotherapy` | minutes | numeric, training volume |
| `Plyometric training` | minutes | numeric, training volume |
| `Sport-specific training` | minutes | numeric, training volume |
| `Other training` | minutes | numeric, training volume |

**Three of those label columns are ordinal, not merely categorical.** `Education`
runs Elementary school -> Middle school -> High school -> University Degree ->
Post Graduate; `Income` runs Low -> Middle-Low -> Middle -> Middle-High -> High;
`Age group` runs 0-35 -> 35-55 -> 55<=. The levels have an order, and one-hot
encoding discards it: encoded that way, "High" and "Low" are two different
labels, as unrelated as two brands. An ordinal encoding keeps the order and costs
one column instead of five. This notebook one-hots them, which is the safe default
when you do not know that the spacing between bands is meaningful.

`Age group` also carries a fourth level, **`0`** which is a placeholder someone typed.

In [ ]:
import sys
import warnings
from collections import namedtuple
from functools import partial
from pathlib import Path

# course_helpers.py sits in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import (
    MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler,
)

# course_helpers.py is the course toolbox: one file, kept beside every
# notebook, holding what is the same every week. It contains no machine
# learning, only the two plot colours and CleaningLog, a small recorder for the
# decisions made below. It is short; open it.
from course_helpers import PLOT_BLUE, PLOT_ORANGE, CleaningLog

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # For reproducibility
FIGSIZE = (10, 6)

# One warning class is silenced here, on purpose. Read the next cell before
# deciding whether that is ever a reasonable thing to do.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

<div class="alert alert-block alert-warning">

**The one warning this notebook silences.**

The comparisons below fit a logistic regression on the columns exactly as they
are, and `Recovery` runs to 9,666 while `Late enrollment` is 0 or 1. On that
spread the solver cannot finish inside its iteration budget, so it raises a
`ConvergenceWarning`, **250 of them** over this notebook, burying every table it
prints.

Silencing a warning is not answering it. Three things make this one defensible,
and all three are needed:

1. The cause is **named**: the columns are on wildly different scales.
2. The fix is **known**, and belongs to a later week.
3. The consequence is **measured**: the worry is that a treatment which shrinks
   the scale wins only by helping the solver, and the scaling section measures
   exactly that on identical splits.

If you cannot do all three, leave the warning printing.

</div>

__Step 1:__ Open the file and look at what has to change.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

In [ ]:
# One file in. `RecordID` identifies the row, so it goes in the index rather
# than sitting among the columns a model could be handed by accident.
raw = pd.read_csv("../../data/raw/champions_raw.csv", index_col="RecordID")

print(f"{raw.shape[0]:,} rows x {raw.shape[1]} feature columns, indexed by RecordID")
raw.head()

# <font color='#E8800A'>Structural survey</font> <a class="anchor" id="survey"></a>
[Back to TOC](#toc)

Before any statistic, two questions: what is
each column *for*, and what would it cost to feed the frame to a model as it
stands?

__Step 2:__ Sort the columns by what they hold. The rest of the notebook
applies different rules to each group.

In [ ]:
# A column distinct on more than half its rows identifies the row, not a
# property of it.
id_like = [c for c in raw.columns if raw[c].nunique() > 0.5 * len(raw)]

# Sort every column into a role. Nothing below treats a group it did not name.
identifiers = id_like
target = "Outcome"
numeric = [c for c in raw.columns
           if c not in identifiers + [target]
           and pd.api.types.is_numeric_dtype(raw[c])]
categorical = [c for c in raw.columns
               if c not in identifiers + [target] + numeric]

# A dtype is how a value is stored, not what it measures, so the numeric group
# is checked for columns with few distinct values: those are labels written as
# numbers, and nothing stops a model treating the codes as quantities.
low_cardinality = {c: raw[c].nunique() for c in numeric if raw[c].nunique() <= 10}
levels = raw[categorical].nunique().sort_values(ascending=False)

# How many levels a column has says how wide encoding will get. WHICH levels it
# has says whether the column means what its name says, and that is a different
# question with a cheaper answer.
for column in levels[levels <= 4].index:
    print(f"{column:20s} {sorted(raw[column].dropna().unique())}")

roles = pd.Series({
    "identifiers": f"{len(identifiers)} {id_like}",
    "numeric": f"{len(numeric)}, of which low-cardinality: {low_cardinality}",
    "categorical": f"{len(categorical)}, holding {int(levels.sum())} levels"
                   f" -> {int(levels.sum())} columns once encoded",
    "target": target,
    "'Athlete Id' passes select_dtypes('number')":
        "Athlete Id" in raw.select_dtypes("number").columns,
})
print(roles.to_frame("what each group holds").to_string())
levels.to_frame("levels")

**Twelve categorical columns hold 49 levels between them**, with `Region`
(13) and `Competition` (7) carrying most of them. Encoding turns levels into
columns, so 49 is the width those twelve would become. How the encoding is done
comes later in the course; the count is a structural fact about the file.

Two of those columns answer with more levels than they have. `Mental
preparation` holds `TRUE`, `FALSE` and **`FASE`**, and `Age group` holds `0`
beside its three bands. Neither is a third state of anything: one is a
misspelling and the other is a placeholder. A null count would not have found
either, because nothing is missing, and a distribution table would not either,
because neither column has a distribution. Listing the levels finds both in one
line.

**The dtype split is a starting point, not the answer.** It is wrong in
both directions here.

Six of the twelve categoricals are **booleans written as text**. And two of the numeric columns are **categories
stored as numbers**: `Edition` is a label for which running of the event it was,
and `Previous attempts` is a count with seven distinct values. `int` and `float`
carry no promise that arithmetic on the column means anything, so a model handed
`Edition` will happily treat 2020 as twice 1010.

The low-cardinality check above is the cheap way to spot them: a numeric column
with a handful of distinct values is usually a label. It is a prompt to look, not
a verdict, and what a column measures is a question about the file, which the
table at the top of this notebook answers.

**`Athlete Id` has 4,000 distinct values and is stored as a number**, so
`select_dtypes("number")` hands it to a model as a feature, a 4,000-value
feature whose only content is *which athlete this is*. **It is an
identifier, and this notebook never lets it into a model matrix.** `RecordID` was
the same trick with 4,280 values, which is why it went into the index at Step 1
and is not in this table at all.

__Step 3:__ Count repeated rows three ways before computing anything else.
Every statistic below is a count or a sum over rows, so a row that appears twice
is counted twice in all of them.

In [ ]:
# `RecordID` is in the index, so `.duplicated()` already ignores it
duplicate_rows = raw.duplicated().sum()
print("duplicates ignoring the index:", duplicate_rows)

per_athlete = raw.groupby("Athlete Id").size()
print(f"\nsurplus rows on 'Athlete Id': {raw.duplicated(subset='Athlete Id').sum()}")
print(f"rows sharing an 'Athlete Id': {per_athlete[per_athlete > 1].sum()}"
      f" across {(per_athlete > 1).sum()} athletes")
print("records per athlete:", per_athlete.value_counts().to_dict())

**The duplicate check returns 0.** Every
row in this file is unique, so an exact-duplicate check finds nothing. That is
not the same as the file having no repetition, and stopping at this check is how
the repetition here would be missed entirely.

The repetition is one level up: **553 rows belong to 273 athletes who appear more
than once**, 266 athletes twice and 7 three times, which is **280 surplus rows**.
Two numbers, two questions. `duplicated(subset="Athlete Id")` returns **280**, the
rows that would go; **553** is how many rows are *involved*. Quote the wrong one
and your log says something false.

__Step 4:__ Look at the target itself, before looking at anything against it.
Every choice further down is judged by a score, and which score to report
depends on this one number.

In [ ]:
balance = raw["Outcome"].value_counts().sort_index()
print(balance.to_frame("rows").assign(
    share=lambda f: (f["rows"] / len(raw)).round(4)).to_string())

# The floor any classifier clears by refusing to think: always answer with
# whichever class is larger.
print(f"\nalways predicting the majority class scores"
      f" {balance.max() / balance.sum():.4f} accuracy and 0 recall on the other")

**2,555 winners against 1,725 non-winners, a 59.70% majority.**
That is close enough to balanced that accuracy is readable, and far enough from
it that accuracy alone would still flatter a model: answering "won" every time
scores 0.5970 and finds none of the 1,725. This week reports **F1** for that
reason, and the number to keep is the 0.5970, because a treatment that moves the
class balance has changed the question rather than cleaned the data.

__Step 5:__ Ask where the missing values are, and along which axis.

In [ ]:
rows_with_null = raw.isna().any(axis=1)
per_column = raw.isna().sum()

pd.Series({
    "rows with a null": f"{rows_with_null.sum():,} of {len(raw):,}"
                        f" ({100 * rows_with_null.mean():.2f}%)",
    "columns with a null": f"{(per_column > 0).sum()} of {raw.shape[1]}",
    "null cells": f"{raw.isna().sum().sum():,}",
    "worst column": f"{per_column.idxmax()} ({per_column.max()} nulls)",
}).to_frame("missing values as the file arrives")

**527 rows of 4,280 (12.31%) carry a null, and the nulls are spread
across 26 of the 28 columns**: 560 cells in total, with no column worse than
`Sex` at 30.

That is the whole of the *which axis* question, and neither answer is free.
Dropping every incomplete row costs 12.31% of the data. Dropping every column
that contains a null costs **26 of 28 columns**: the dataset. The missing values are
scattered rather than concentrated, so there is no cheap corner to cut.

# <font color='#E8800A'>Distribution statistics</font> <a class="anchor" id="distributions"></a>
[Back to TOC](#toc)

Deepening this week's exploration means going **both visual and
statistical**. This is the statistical half, and it decides which summary the
next section is allowed to use.

__Step 6:__ Describe the eleven training-volume columns.

In [ ]:
# Eleven of the numeric columns are one family: a duration in minutes,
# recorded the same way. They are summarised together because they are the
# same measurement, and later weeks transform them together too.
training_minutes = [c for c in numeric if c not in ["Edition", "Previous attempts",
                                            "Athlete score"]]

# .describe() returns a DataFrame with one COLUMN per input column. Transpose
# it and each row is a column of the file, which is the orientation to read
# eleven of them in.
summary = raw[training_minutes].describe().T
print(summary.round(2).to_string())

`describe()` reports count, mean, std, min, the three quartiles and
max. Two things it does not report decide how the rest of this notebook reads
these columns: how **skewed** each one is, and how much of it is exactly
**zero**. The `50%` row is the median, so the mean-to-median ratio is already
available from what is printed above.

What comes back from `describe()` is an ordinary DataFrame, so the missing
statistics can be added to it as columns rather than assembled into a second
table.

__Step 7:__ Add the statistics `.describe()` leaves out to that same frame.

In [ ]:
shape = summary.copy()
# The 50% column IS the median, so the ratio comes from the frame itself.
# A median of zero leaves it undefined rather than infinite.
shape["mean/median"] = shape["mean"] / shape["50%"].replace(0, np.nan)
shape["skew"] = raw[training_minutes].skew()
shape["% zeros"] = 100 * (raw[training_minutes] == 0).mean()
shape = shape.sort_values("skew", ascending=False)

print(shape[["mean", "50%", "mean/median", "skew", "% zeros"]].round(2).to_string())
print("\nratio defined for", shape["mean/median"].notna().sum(), "of", len(shape),
      f"columns, spanning {shape['mean/median'].min():.2f}-{shape['mean/median'].max():.2f}")

**Where the ratio is defined at all, the mean is 1.57 to 3.06 times
the median**: `Cardiovascular training` 266.45 against 87, `Recovery` 307.94
against 117. Every one of these columns is right-skewed, and the mean is sitting
out in the tail where almost no athlete is.

**The ratio is defined for only 6 of the 11 columns**, and that is the second
finding. `Sand training`, `Squad training`, `Physiotherapy`, `Plyometric
training` and `Other training` all have a median of **0**, because most athletes
do none of them: 76.14% of `Sand training` is zero, 82.41% of `Plyometric
training`. You cannot divide by that, and you cannot
run a boxplot rule over it either.

`Sand training`'s skew of **27.29** comes from the fact that it is 76% zeros with a maximum
of 1,213.

The same skew as a picture: the histogram with the median and the mean
drawn on it.

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)
sns.histplot(raw["Recovery"].dropna(), bins=60, color=PLOT_BLUE, ax=ax)
ax.axvline(raw["Recovery"].median(), color=PLOT_ORANGE, lw=2,
           label=f"median {raw['Recovery'].median():.0f}")
ax.axvline(raw["Recovery"].mean(), color="black", lw=2, ls="--",
           label=f"mean {raw['Recovery'].mean():.0f}")
ax.set(xlabel="Recovery (minutes)", ylabel="athletes",
       title=f"Recovery: skew {raw['Recovery'].skew():.2f}")
ax.legend()
plt.show()

**Median 117, mean 308.** The median sits inside the bar where most
athletes are; the mean sits in empty space to its right, dragged there by a
handful of athletes recovering for thousands of minutes.

# <font color='#E8800A'>An invalid value is not always missing data</font> <a class="anchor" id="invalid"></a>
[Back to TOC](#toc)

Some cells in this file are not extreme. They are impossible, and
what you do about that is a choice, decided by the column's *domain* before you
look at any result.

<div class="alert alert-block alert-info">

**The rule, stated before the measurement.** Every column has a set
of values it is allowed to take, fixed by what it measures:

| column type | domain | what a value outside it is |
|---|---|---|
| a duration in minutes | $[0, \infty)$ | not a small value, **not a value** |
| a score on a 0-140 scale | $[0, 140]$ | the same |
| a boolean | exactly two levels | a third level is a defect, not a category |

An impossible value carries no information about the athlete, so treating
it as a number lets it move a mean, a fence or a coefficient. **It cannot stay a
number**, and what it becomes instead is the choice this section measures.

**Dropping the row is the last resort, not the first.** One bad cell is one cell;
the row still carries 28 good ones, and the rows with defects are rarely a random
sample of the file, so deleting them changes what the dataset is *about*.

**The caveat.** Recording these cells as `NaN` makes the dataset *defensible*: it
stops a code being read as a quantity. It does not make it more **accurate**, and
it is not automatically the better choice either. The value becomes a missing value that
something now has to fill, and the fill is a guess of its own, made by you rather
than by whoever typed the code. Where the code itself carries information, keeping
it as an explicit category can beat erasing it.

</div>

__Step 8:__ Find the columns whose domain forbids a negative.

In [ ]:
non_negative = ["Physiotherapy", "Athlete score"]
for column in non_negative:
    print(f"{column:16s} min {raw[column].min():7.1f}   negative rows"
          f" {(raw[column] < 0).sum():4d}")

step3 = raw.copy()
step3[non_negative] = step3[non_negative].mask(step3[non_negative] < 0)
print(f"\nrows with a null: {rows_with_null.sum():,} ->"
      f" {step3.isna().any(axis=1).sum():,}"
      f" ({100 * step3.isna().any(axis=1).mean():.2f}% of {len(step3):,})")
print(f"null cells:       {raw.isna().sum().sum():,} -> {step3.isna().sum().sum():,}")

**527 rows become 954 (22.29%), and 560 null cells become 1,069.**
Recording invalid values as missing makes the missing values go up, not down.

Those counts are for the two columns whose domain forbids a negative.
`Physiotherapy` contributes 4 cells and `Athlete score` **505**; `Age group`'s
`0` band is the third rule and adds 4 more, for 513 in total across 510 rows.
And those 505 are not what they look like.

__Step 9:__ Look at the negative scores by value. Random damage scatters across
many values; a placeholder value piles up on a few, so the counts begin to say
which kind of missing value this will become.

In [ ]:
score_placeholder = raw["Athlete score"] < 0
raw.loc[score_placeholder, "Athlete score"].value_counts().to_frame(
    "rows at this value")

**Where did the missing values come from?** The file arrived with 560 blank
cells across 527 rows and 26 columns. The domain rule then turns 513 impossible
values into further missing values. Those are two different sources, so the analysis keeps
two row flags: a row with any blank as shipped, and a row carrying the negative
score code.

Whether either flag is missing at random has a precise, standard answer with
three cases, and which one holds decides whether a fill can be trusted:

- **MCAR**, missing completely at random: whether a cell is missing has nothing to
  do with anything -- not the hidden value, not any other column. The missing values are a
  random sample, and a simple fill costs only precision.
- **MAR**, missing at random: whether a cell is missing depends on *other,
  observed* columns, but not on the hidden value itself once those are accounted
  for. A fill informed by those columns is defensible.
- **MNAR**, missing not at random: whether a cell is missing depends on the hidden
  value itself, or on something never recorded. No fill is safe on its own, and an
  explicit "was missing" flag often beats erasing the pattern.

You never see the hidden values, so MCAR cannot be proven outright. What you *can*
do is test each missingness flag against the columns you *do* observe: if a flag
is tied to any of them, those missing values are not completely at random, and the fill has
to answer for that.

**The target is left out of these tests on purpose.** Deciding how to clean the data
from `Outcome` would let the label steer preprocessing -- a leak, one step earlier
than fitting a model to it. Missingness is diagnosed from the **features only**.

__Step 10:__ Cross each missing value flag with each categorical feature and ask one question:
does the flag fall evenly across that feature's levels, or does it pile up on
some of them?

Both flags are per ROW, not per column. `any column blank` marks a row carrying
at least one blank among the 28 columns; `negative score placeholder` marks a row
whose `Athlete score` holds the negative code. They are pooled across columns
because they have to be: no single column carries more than 30 blanks in 4,280
rows, which leaves every per-column test with expected counts near zero and
nothing to measure.

Three numbers come out of each crossing, and they do different jobs.
**Chi-square** is the distance between the counts observed and the counts an
evenly-spread flag would produce. It grows with the size of the table and with
the number of rows, so it ranks features within one flag and cannot be compared
across them. **Cramér's V** rescales that distance onto 0 to 1, where 0 is a flag
spread exactly as chance would spread it and 1 is a feature that fixes the flag
completely, so V is the number to compare. **The minimum expected count** guards
both: the chi-square approximation needs roughly 5 expected observations in every
cell of the table, and below that the number it returns is not interpretable
however small the p-value looks.

In [ ]:
missing_flags = {
    "any column blank": raw.isna().any(axis=1),
    "negative score placeholder": score_placeholder,
}
rows = []
for missing_source, flag in missing_flags.items():
    for column in categorical:
        table = pd.crosstab(flag, raw[column])
        chi2, p, _, expected = chi2_contingency(table)
        rows.append({
            "missing source": missing_source,
            "feature": column,
            "chi2": chi2,
            "Cramer's V": association(table, method="cramer", correction=True),
            "p": p,
            "min expected count": expected.min(),
        })
association_table = pd.DataFrame(rows)

# One row per flag, one column per feature, coloured by V. Twenty-four numbers
# are quicker to read as a grid than as a twenty-four-row table.
strength = association_table.pivot(
    index="missing source", columns="feature", values="Cramer's V")
too_thin = association_table.pivot(
    index="missing source", columns="feature", values="min expected count") < 5
by_strength = strength.loc["negative score placeholder"].sort_values(
    ascending=False).index
strength, too_thin = strength[by_strength], too_thin[by_strength]

fig, ax = plt.subplots(figsize=(11, 3.2))
sns.heatmap(strength, vmin=0, vmax=1, annot=True, fmt=".3f", linewidths=2,
            cmap=sns.light_palette(PLOT_BLUE, as_cmap=True),
            cbar_kws={"label": "Cramér's V"}, ax=ax)
# Hatched where the minimum expected count is under 5: that cell's colour is
# arithmetic rather than evidence, and hatching says so without hiding it.
for row, column in zip(*np.where(too_thin.values)):
    ax.add_patch(plt.Rectangle((column, row), 1, 1, fill=False, hatch="///",
                               edgecolor=PLOT_ORANGE, linewidth=0))
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Is the missing value flag tied to the feature? (hatched: too thin to trust)")
plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

**The blanks already in the source are broadly scattered.** None
of their twelve categorical tests survives the conservative threshold
$0.05 / 12 = 0.0042$. `Region` has the largest credible Cramér's V at only
**0.0634** with p = **0.1456**. `Competition` reaches an unadjusted p = **0.0386**,
but its V is **0.0559**. That is weak evidence after twelve tests, not a reason to
build a special fill around one category.

**The negative score code is different.** `Competition` gives chi-square
**2457.91** and Cramér's V **0.7596**, so the missing values this code becomes are not
missing completely at random. Whether the score was recorded with the code
depends heavily on which event the row belongs to.

The hatched cells are the ones no p-value can rescue. `Age group`'s smallest
expected cell is **0.47**, below the rule-of-thumb floor of 5, so its chi-square
cannot be trusted however small its p-value looks.

__Step 11:__ Read the placeholder rate event by event, so the association above
becomes a concrete pattern rather than a single number.

In [ ]:
by_competition = (score_placeholder.groupby(raw["Competition"]).mean() * 100).round(2)
print("share of rows carrying the negative score placeholder, by competition (%):")
print(by_competition.sort_values(ascending=False).to_string())

<div class="alert alert-block alert-warning">

**These are placeholder values, not random corruption.** Random damage does not
land on one value: **499 of the 505 negatives are exactly −30**, with 4 at −15
and 2 at −20. Three round numbers, one of them 99% of the cases, is a code
somebody entered (most likely "not assessed") written into a numeric column.

**And the code is administrative, not random.** It appears in only 3 of the 7
competitions -- **92.28%** of World Championship rows, **23.36%** Continental
Championship, **23.03%** National Cup, and **0%** everywhere else. That is what
"not missing completely at random" looks like on this file: the missing value is tied to the
event, an observed feature, so the missingness carries a structure a blind fill
could erase.

**The recipe still imputes it, and the fill has a cost.** Filling those 505 cells
with the column median -- which is 0 -- turns every one of them into another zero,
deepening the zero-spike in a column every model downstream reads. An explicit
`Athlete score_was_missing` indicator is a defensible alternative.

</div>

Put the whole missingness table into one picture: every column's nulls as
shipped and after the domain rule.

In [ ]:
before = raw.isna().sum()
after = step3.isna().sum()
order = after.sort_values(ascending=False).index

# Both series in one long frame, which is the shape seaborn plots from:
# one row per (column, when), and `hue` does the rest.
# The taller series first, so the shorter one draws in front of it. Overlaid
# bars only tell you anything if the one that grew is behind the one it grew
# from; the other order hides the whole comparison.
nulls = (pd.DataFrame({"after the domain rule": after, "as shipped": before})
         .loc[order].rename_axis("column").reset_index()
         .melt(id_vars="column", var_name="when", value_name="null cells"))

fig, ax = plt.subplots(figsize=FIGSIZE)
sns.barplot(nulls, x="column", y="null cells", hue="when", dodge=False,
            palette=[PLOT_ORANGE, PLOT_BLUE], ax=ax)
ax.set(xlabel="", title=f"Missingness across all {raw.shape[1]} columns")
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
plt.tight_layout()
plt.show()

Twenty-six numbers in a table show that missing values exist. The picture
shows the *pattern*: a long flat tail of columns missing twenty-odd cells each,
and one orange spike where `Athlete score` jumps from 16 nulls to 521.

# <font color='#E8800A'>Where the boxplot fails</font> <a class="anchor" id="boxplot"></a>
[Back to TOC](#toc)

**What a boxplot is for.** It draws five numbers: the median, the
first and third quartiles as the box, and two whiskers reaching to the furthest
points still within **1.5 × IQR** of the box, where the IQR is the width of the
box itself. Anything past a whisker is drawn as a dot and conventionally called
an outlier.

The appeal is that it is distribution-free: it uses ranks, so one enormous value
cannot drag the box the way it drags a mean. That makes it the standard first
tool for finding outliers, and on a column with a middle it works well.

The assumption hiding in it is that the box has width.

Two figures follow, one column each. The **blue** figure is the column the rule
suits and the **orange** figure is the column it does not, so the colour marks
which case you are looking at rather than which panel.

__Step 12:__ Draw the 1.5 x IQR rule twice: once on a column that has a
middle, and once on a column that does not.

In [ ]:
# First, the rule on a column it was designed for. `Recovery` has a middle:
# the box has width, and the dots past the whiskers are a few very high
# values out of four thousand.
fig, (left, right) = plt.subplots(1, 2, figsize=FIGSIZE)
sns.histplot(raw["Recovery"].dropna(), bins=60, color=PLOT_BLUE, ax=left)
left.set(xlabel="Recovery (minutes)", ylabel="athletes",
         title="a column with a middle")
sns.boxplot(x=raw["Recovery"].dropna(), color=PLOT_BLUE, ax=right)
right.set(xlabel="Recovery (minutes)", title="and the boxplot agrees")
plt.tight_layout()
plt.show()

recovery = raw["Recovery"].dropna()
r_q1, r_q3 = recovery.quantile([0.25, 0.75])
r_iqr = r_q3 - r_q1
r_flagged = (~recovery.between(r_q1 - 1.5 * r_iqr, r_q3 + 1.5 * r_iqr)).sum()
print(f"Recovery: Q1 {r_q1:.0f}, Q3 {r_q3:.0f},"
      f" {r_flagged:,} of {len(recovery):,} rows flagged")

# Now the same rule on `Sand training`. Both panels of this figure are orange,
# and both panels of the one above are blue: the colour marks the CASE, the
# column the rule suits against the column it does not, so one column is never
# drawn in two colours.
fig, (left, right) = plt.subplots(1, 2, figsize=FIGSIZE)
sns.histplot(raw["Sand training"].dropna(), bins=60, color=PLOT_ORANGE, ax=left)
left.set(xlabel="Sand training (minutes)", ylabel="athletes",
         title="what the column is")
sns.boxplot(x=raw["Sand training"].dropna(), color=PLOT_ORANGE, ax=right)
right.set(xlabel="Sand training (minutes)",
          title="what the boxplot says it is")
plt.tight_layout()
plt.show()

sand = raw["Sand training"].dropna()
q1, q3 = sand.quantile([0.25, 0.75])
iqr = q3 - q1
flagged = (~sand.between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)).sum()
print(f"Q1 {q1:.0f}, Q3 {q3:.0f}, IQR {iqr:.0f}  ->  {flagged:,} rows flagged"
      f" ({100 * (raw['Sand training'] == 0).mean():.2f}% of the column is zero)")

<div class="alert alert-block alert-warning">

**Read the two panels together.** The histogram shows a column where
three athletes in four do no sand training at all and the rest do between a
little and a lot. The boxplot shows a flat line at zero with a cloud of dots
above it, every one of them labelled an outlier.

**Q1 = 0, Q3 = 0, so the IQR is 0, and a fence of zero width flags every
non-zero value.** That is **1,002 rows**, a quarter of the file, declared
anomalous by a rule that has not looked at a single one of them.

Nothing is wrong with the rule's arithmetic. The rule assumes a distribution with
a middle, and this column has no middle: it is a spike at zero with a tail. **A
diagnostic is only valid where its assumption holds**, and "76% of the values are
identical" is where this one stops holding. The same is true of `Other training`
(76.33% zeros), `Plyometric training` (82.41%) and `Squad training` (58.15%).

</div>

# <font color='#E8800A'>The duplicates `drop_duplicates` misses</font> <a class="anchor" id="duplicates"></a>
[Back to TOC](#toc)

The survey asked whether any row is repeated and got a flat no. That
answer is about rows. The question a dataset actually turns on is whether any
*athlete* is repeated, and the two come apart here.

The **280 surplus rows** matter for a reason no count reveals: an athlete on both sides of a
train/test split puts the same person's characteristics in the training data and
in the exam, and every score measured afterwards is flattered by it.

Removing the surplus needs a **stated tie-break**, because "whichever row came
first in the file" is not a rule; it depends on how the file was shuffled. Which
row to keep is a decision, so it is one of the things this exploration hands
forward to be measured rather than settled here.

# <font color='#E8800A'>From exploration to a recipe</font> <a class="anchor" id="plan"></a>
[Back to TOC](#toc)

The exploration is over. Nothing above changed a single cell of `raw`, and that was the point. What it
can do is leave you a list.

<div class="alert alert-block alert-info">

**What the exploration found, and what is still undecided.**

Six findings, and each one names a question the exploration could not answer,
because answering it means changing the file and the only way to compare two
changes is to measure them.

| # | what the exploration found | what is still open |
|---|---|---|
| 1 | Roles, not dtypes, decide treatment. Booleans arrive as text, `Edition` and `Previous attempts` arrive as numbers, `Athlete Id` is a 4,280-value identifier a numeric filter would hand to a model. | Nothing. The four roles are fixed by what each column measures, and no measurement changes that. |
| 2 | `Mental preparation` answers with three levels, `TRUE`, `FALSE` and `FASE`, and `Age group` with a `0` beside its three bands. | Whether to repair the misspelling, blank it, or drop those rows. |
| 3 | Some cells are impossible rather than extreme: negative durations and negative scores, and the negative score is an administrative placeholder tied to the event. | Whether an impossible value should be blanked, clipped to the nearest allowed value, dropped with its row, or kept and flagged. |
| 4 | `drop_duplicates()` finds nothing, yet 280 surplus rows belong to 273 athletes who appear more than once. | Which row of a repeated athlete to keep. |
| 5 | 26 of 28 columns carry missing values, scattered, so neither dropping rows nor dropping columns is affordable. | Which fill, and whether filling beats dropping at all. |
| 6 | The training-volume columns are right-skewed, mean over median 1.57 to 3.06, several more than three-quarters zeros, and the boxplot rule inverts on them because `Q1 = Q3` makes the fence zero-width. | Which outlier treatment earns its place. |

**Six open questions, and every one of them gets measured.** That is what the
rest of the notebook is: one section per question, each one running the
alternatives through the same function the recipe will call, and each one ending
in a choice written down as a value rather than as a sentence.

</div>

# <font color='#E8800A'>Impossible values: blank, clip or drop</font> <a class="anchor" id="invalidfix"></a>
[Back to TOC](#toc)

The first of the five open questions, and the one the rest depend
on, because every later comparison runs on whatever this one decides. A negative
physiotherapy duration is not a large value; it is a value the column's own units
forbid. What it should become is a separate question from what it is.

<div class="alert alert-block alert-info">

**How every comparison in this notebook is scored, stated once.**

1. The frame is split into training and test rows.
2. The treatment (an imputer, a fence, a transform) is **fitted on the training
   rows only**. A median computed over rows the model is about to be tested on is
   the test set leaking into the training data.
3. Rows may be dropped from the **training** side. Never from the test side: a
   candidate that deletes its own hard test rows is grading its own exam.
4. Every candidate is scored on **identical, untouched test rows**.
5. Repeat over 20 splits and report the mean with a standard error, because a
   single split of 4,000 rows moves these numbers by more than the effects being
   compared.

The model is a plain logistic regression. It is a **measuring instrument**
here, not a result; the model itself becomes the subject later in the course.

</div>

__Step 13:__ Set up what every comparison below runs through: the model matrix, the
fill that makes a split ready for a model, and the scorer. They are written
once and never again.

In [ ]:
identifiers = ["Athlete Id"]
numeric = [c for c in raw.columns
           if c not in identifiers + ["Outcome"]
           and pd.api.types.is_numeric_dtype(raw[c])]
categorical = [c for c in raw.columns
               if c not in identifiers + ["Outcome"] + numeric]


def design_matrix(frame, numeric, categorical):
    """Numbers as they are, everything else one-hot. Identifiers never enter."""
    encoder = OneHotEncoder(sparse_output=False, dtype=float)
    encoded = encoder.fit_transform(frame[categorical].astype("string"))
    text = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(categorical),
        index=frame.index,
    )
    return pd.concat([frame[numeric], text], axis=1)


def fill_missing(train, test, plan):
    """Fill every missing value with imputers fitted on `train` alone.

    `plan` is a list of (columns, imputer) pairs, and any scikit-learn imputer
    works: each one is fitted on the training rows of its own columns and fills
    those columns in both halves, so the test rows never help compute a fill.
    """
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        # scikit-learn reads np.nan as a gap but not pd.NA, the gap of a
        # nullable column such as the booleans, so every gap becomes np.nan.
        blocks = [half[columns].astype(object).fillna(np.nan).infer_objects()
                  for half in (train, test)]
        if isinstance(imputer, (KNNImputer, IterativeImputer)):
            # KNN measures distances between rows, and the model inside
            # IterativeImputer can be any regressor, many of them sensitive to
            # scale. So both work on standardised columns, and their fills go
            # back in the columns' own units.
            scale = StandardScaler().fit(blocks[0])
            imputer.fit(scale.transform(blocks[0]))
            filled = [scale.inverse_transform(imputer.transform(scale.transform(block)))
                      for block in blocks]
        else:
            imputer.fit(blocks[0])
            filled = [imputer.transform(block) for block in blocks]
        train[columns], test[columns] = filled
    return train, test


# A model cannot fit a missing value, so every comparison fills inside its own
# split. Until the missing-values section chooses a plan, that fill is the
# training median for the numbers and the training mode for the labels.
median_mode = [(numeric, SimpleImputer(strategy="median")),
               (categorical, SimpleImputer(strategy="most_frequent"))]

Scored = namedtuple("Scored", "f1 iterations")


def sem(values):
    """Standard error of the mean: how far it would move on new splits."""
    return values.std(ddof=1) / np.sqrt(len(values))


def held_out_f1(prepare, frame, splits, numeric, categorical, target="Outcome",
                scaler=None):
    """Score one treatment by F1 on each split's untouched test rows.

    `prepare(train, test)` applies the treatment, fitted on `train` only. The
    result holds the test-row F1 per split (`.f1`) and the solver's iterations
    (`.iterations`). `scaler`, when given, is fitted on the training half and
    applied to both.
    """
    f1, iterations = [], []
    for train_rows, test_rows in splits:
        train, test = prepare(frame.iloc[train_rows], frame.iloc[test_rows])
        matrix = design_matrix(pd.concat([train, test]), numeric, categorical)
        a, b = matrix.iloc[:len(train)], matrix.iloc[len(train):]
        if scaler is not None:
            fitted = scaler.fit(a)
            a, b = fitted.transform(a), fitted.transform(b)
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(a, train[target])
        iterations.append(int(model.n_iter_[0]))
        f1.append(f1_score(test[target], model.predict(b)))
    return Scored(np.array(f1), np.array(iterations))


print(f"{len(numeric)} numeric columns, {len(categorical)} categorical,"
      f" {len(identifiers)} identifiers held out of the model")

__Step 14:__ Draw the twenty stratified 80/20 splits every candidate will be judged on,
so the comparison is over the same test rows twenty times rather than one lucky
split.

In [ ]:
# The six booleans arrive as text, and one of them is misspelt. Each spelling
# maps straight to the value the column should hold.
boolean_columns = ["Mental preparation", "Disability", "Late enrollment",
                   "Outdoor Workout", "No coach", "Past injuries"]
text_fixes = {"FASE": False, "FALSE": False, "TRUE": True}

# Twenty independent 80/20 draws that keep the class balance of the whole
# frame, so every candidate below is judged on the same twenty test sets. A
# comparison draws its splits on the frame it is comparing ON.
splits = StratifiedShuffleSplit(n_splits=20, test_size=0.2,
                                random_state=RANDOM_STATE)

# The text repair comes first and is not one of the alternatives: leaving
# `FASE` as a third level of a two-level column is not a treatment anybody would
# defend. Everything from here runs on the repaired frame.
repaired = raw.copy()
repaired[boolean_columns] = raw[boolean_columns].astype("object").replace(text_fixes)
repaired_splits = list(splits.split(repaired, repaired["Outcome"]))
print(f"{len(repaired_splits)} splits on {len(repaired):,} rows;"
      f" `Mental preparation` now has"
      f" {repaired['Mental preparation'].nunique()} levels, not 3")

__Step 15:__ Score the four ways of treating an impossible value against each
other. Three of them go through `fix_invalid` with a different replacement; the
fourth drops the row, which is not something a fix can do, so it is built at the
row level and dropped from the training side only.

In [ ]:
# The rule for each column whose domain forbids a value: True for an
# impossible one.
invalid_errors = {
    "Physiotherapy": lambda values: values < 0,
    "Athlete score": lambda values: values < 0,
    "Age group": lambda values: values == "0",
}
invalid_cols = list(invalid_errors)
numeric_invalid = ["Physiotherapy", "Athlete score"]


def domain_flags(frame, cols, errors):
    """One boolean column per rule, True where that column's value is impossible."""
    return pd.concat([errors[column](frame[column]) for column in cols], axis=1)


def fix_invalid(frame, cols, errors, replacement):
    """Every impossible value in `cols` becomes `replacement`.

    `np.nan` blanks the cell, recording that the value is unknown; `0` clips it,
    claiming the value was meant to be zero. Which is right is measured below.
    """
    out = frame.copy()
    out[cols] = out[cols].mask(domain_flags(frame, cols, errors), replacement)
    return out


# Each arm is a (train, test) -> (train, test) function, the shape every
# comparison in this notebook uses, and each fills what it leaves missing.
def treat_invalid(train, test, plan):
    """Apply each (columns, replacement) pair of `plan`, then fill."""
    for cols, replacement in plan:
        train = fix_invalid(train, cols, invalid_errors, replacement)
        test = fix_invalid(test, cols, invalid_errors, replacement)
    return fill_missing(train, test, median_mode)


def drop_invalid_rows(train, test):
    """Drop the offending rows from the TRAINING half, then fill the rest."""
    return fill_missing(
        train[~domain_flags(train, invalid_cols, invalid_errors).any(axis=1)],
        test, median_mode)


def flag_invalid_rows(train, test):
    """Leave the impossible value in place and record it in its own column."""
    train, test = (pd.concat([f, domain_flags(f, invalid_cols, invalid_errors)
                              .add_suffix(" out of domain")],
                             axis=1) for f in (train, test))
    return fill_missing(train, test, median_mode)


arms = {
    "blank the cell": partial(treat_invalid,
                              plan=[(invalid_cols, np.nan)]),
    # `Age group`'s impossible value is the string "0", which has no numeric
    # floor to clip to, so this arm clips the two numeric columns and blanks
    # that one.
    "clip to zero": partial(treat_invalid,
                            plan=[(numeric_invalid, 0),
                                  (["Age group"], np.nan)]),
    "drop the row": drop_invalid_rows,
    "keep and flag": flag_invalid_rows,
}
# The flag arm adds columns, so its scorer needs the longer categorical list.
flag_columns = domain_flags(repaired, invalid_cols, invalid_errors).add_suffix(
    " out of domain")
flagged_categorical = categorical + list(flag_columns.columns)
invalid_scores = {
    name: held_out_f1(fn, repaired, repaired_splits, numeric,
                      flagged_categorical if name == "keep and flag" else categorical).f1
    for name, fn in arms.items()
}
control = invalid_scores["blank the cell"]

# One row per arm: its own mean, and its difference from the control arm
# measured on the same twenty splits.
invalid_board = {}
for name, scores in invalid_scores.items():
    difference = scores - control
    invalid_board[name] = {
        "F1": scores.mean(),
        "vs blank": difference.mean(),
        "paired SEM": sem(difference),
        "beats blank on": f"{int((difference > 0).sum())} of {len(difference)}",
    }
print(pd.DataFrame.from_dict(invalid_board, orient="index").round(4).to_string())

# What the drop arm is actually spending, which the F1 column alone does not say.
flagged_rows = flag_columns.any(axis=1)
print(f"\nrows carrying an impossible value: {flagged_rows.sum():,}"
      f" of {len(repaired):,}"
      f"   win rate {repaired.loc[flagged_rows, 'Outcome'].mean():.4f}"
      f" against a base rate of {repaired['Outcome'].mean():.4f}")

<div class="alert alert-block alert-info">

**What the `paired SEM` column is, and why the board carries
one.**

Each arm above is twenty numbers, one per split. The column beside its mean is
the **standard error of the mean**: the spread of those twenty divided by the
square root of twenty. It answers a different question from the mean. The mean
says what an arm scored. 

It is **paired** because it is computed on the differences rather than on the
scores. Every arm is scored on the same twenty splits, so the split-to-split
noise all of them share cancels in the subtraction, and what survives is the
part that actually separates the arms.

Read the two together and never the mean on its own. A difference of -0.0109
with a SEM of 0.0016 is about seven times its own uncertainty and is a result. A
difference of 0.0018 with a SEM of 0.0015 is smaller than the noise it was
measured in, and it should be read as **no difference**, not as "slightly
better".

</div>

**Blanking and clipping score identically. Not closely: identically,
on all twenty splits, to every decimal place the table prints.**

 `Athlete score` has its 25th and 50th percentiles both at zero, so the training median of
the blanked column is 0.0 on every split. The clip arm snaps the impossible cell
to 0 directly.

**Keeping the value and flagging it lands 0.0018 below blanking, with a paired
standard error of 0.0015 and a win on 8 of the 20 splits.** That is a difference
of about one standard error, which is the table's way of saying no difference at
all. The seven extra columns the indicator buys are not paying for themselves
here.

**Dropping the row is the only arm the instrument separates, and it separates
downwards: 0.8060 against 0.8168, a paired difference of -0.0109 ± 0.0016, ahead
on 2 splits out of 20.** Look at the line under the table for why. The 510 rows
carrying an impossible value win at **0.6412 against a base rate of 0.5970**, so
deleting them does not remove noise, it removes winners. A treatment that throws
away 510 of 4,280 rows and moves the class balance while it does so has to show
something for it, and this one shows a loss.

**So: blank the cell.** Not because it won, since nothing won.

__Step 16:__ Apply the treatment that won. `masked` is the repaired frame with its
impossible cells recorded as missing values: nothing filled, and no row removed yet.

In [ ]:
# The treatment this section chose.
chosen_invalid_fix = "blank the cell"

masked = fix_invalid(repaired, invalid_cols, invalid_errors, np.nan)
print(f"{raw.shape[0]:,} rows, {masked.isna().sum().sum():,} null cells"
      f" once the invalid values are blanked, nothing filled or removed yet")

# <font color='#E8800A'>Repeated athletes: which row to keep</font> <a class="anchor" id="tiebreak"></a>
[Back to TOC](#toc)

The exploration found 280 surplus rows belonging to 273 athletes
who entered more than one competition. Removing them is not in question, because
an athlete on both sides of a split hands the model the answer. Which of an
athlete's rows survives is in question, and it is the kind of decision that gets
made by whichever argument `drop_duplicates` happens to default to.

__Step 17:__ Score three tie-breaks, an arm that keeps every row, and five
arbitrary rules. The arbitrary ones are not candidates. They are there to show
how far this score moves for no reason at all, which is the only thing that
makes the other numbers readable.

In [ ]:
def deduplicate(frame, key, choose):
    """Keep one row per `key`; `choose` takes one key's rows and names the survivor."""
    keep = frame.groupby(key, sort=False).apply(choose)
    return frame[frame.index.isin(keep)]


# Each `choose` returns the index label of the row that survives. The rule has
# to be deterministic, so `most complete` breaks its own ties on the earliest
# RecordID.
tiebreaks = {
    "earliest RecordID": lambda rows: rows.index.min(),
    "latest RecordID": lambda rows: rows.index.max(),
    "most complete row": lambda rows: rows.isna().sum(axis=1).sort_index()
                                          .idxmin(),
}
# Five rules with no argument behind them at all, as a yardstick.
for seed in range(5):
    tiebreaks[f"arbitrary (seed {seed})"] = (
        lambda rows, s=seed: rows.sample(1, random_state=s).index[0])

candidates = {name: deduplicate(masked, "Athlete Id", choose)
              for name, choose in tiebreaks.items()}
candidates["keep every row"] = masked


def train_on(train, test, candidate):
    """Swap in the candidate's rows, minus every athlete in the test half, then fill.

    Each candidate keeps a different 4,000 rows, so all of them are scored on ONE
    set of test rows, drawn on the earliest-RecordID frame.
    """
    train = candidate[~candidate["Athlete Id"].isin(test["Athlete Id"])]
    return fill_missing(train, test, median_mode)


reference = candidates["earliest RecordID"]
reference_splits = list(splits.split(reference, reference["Outcome"]))
tiebreak_scores = {
    name: held_out_f1(partial(train_on, candidate=frame), reference,
                      reference_splits, numeric, categorical).f1
    for name, frame in candidates.items()}
earliest_score = tiebreak_scores["earliest RecordID"]

tiebreak_board = {}
for name, frame in candidates.items():
    scores = tiebreak_scores[name]
    difference = scores - earliest_score
    tiebreak_board[name] = {
        "rows": len(frame),
        "missing values": int(frame.isna().sum().sum()),
        "win rate": frame["Outcome"].mean(),
        "F1": scores.mean(),
        "vs earliest": difference.mean(),
        "paired SEM": sem(difference),
    }
print(pd.DataFrame.from_dict(tiebreak_board, orient="index").round(4).to_string())

**Taken at face value the shipped rule wins.** `earliest RecordID`
scores 0.8207, `latest RecordID` and `most complete row` sit at -0.0041 and
-0.0044, and those are two to three times their own standard errors. Keeping
every row costs -0.0008, so the 280 surplus rows carry no information the model
uses, and the reason to remove them is the athlete straddling the split rather
than any gain in score.

Before believing the win, look at the five rules that have no argument behind
them at all. **Picking a random row per athlete scores 0.8184 to 0.8192.** Five
rules chosen for no reason beat both rules chosen for a reason, so the column is
not ranking tie-breaks.

**The tie-break is therefore fixed for reproducibility, not for accuracy.** This
notebook keeps each athlete's earliest `RecordID`, and sorts before it drops so
the result does not depend on the order the file arrived in.

__Step 18:__ Apply the tie-break that was chosen. `deduped` is the deduplicated
frame with its missing values still open, and it is what the next three comparisons run
on.

In [ ]:
chosen_tiebreak = "earliest RecordID"

# Select, do not re-sort: the rows come back in the order the file had them,
# which is what keeps this step from quietly reordering the whole dataset.
deduped = deduplicate(masked, "Athlete Id", tiebreaks[chosen_tiebreak])
deduped_splits = list(splits.split(deduped, deduped["Outcome"]))
print(f"{len(masked):,} rows -> {len(deduped):,} rows"
      f"  ({len(masked) - len(deduped):,} removed),"
      f" {deduped.isna().sum().sum():,} missing values still open")

# <font color='#E8800A'>Missing values: compare nine, then select</font> <a class="anchor" id="imputation"></a>
[Back to TOC](#toc)

Nine ways of filling 1,014 cells, compared on held-out F1, and one of them
chosen.

__Step 19:__ Score nine fill plans on those identical splits.

A **plan** is a list of `(columns, imputer)` pairs, the argument `fill_missing`
already takes, so each group of columns gets its own scikit-learn imputer. Each
name reads *numbers / labels*: `median / mode` fills a missing number with the
training median and a missing label with the most common training label.
**Own level** fills a missing label with a new category of its own,
`"(missing)"`, so the model sees that the label was absent instead of being
told it was the common one. **Proportional** draws each missing label at random
from the training labels, in their training shares, so the column keeps its mix
instead of piling every gap onto the common label; scikit-learn has no such
imputer, so the cell below writes one. One candidate varies the numeric fill by
**skew** rather than by column type. The last two read the other numeric
columns: **KNN** averages the five most similar training athletes, and
**iterative** predicts each column from the rest. They fill numbers only,
because a label has no distance to measure, so the labels keep the training
mode. Both also work on standardised numbers, and `fill_missing` puts their
fills back in the columns' own units: KNN finds neighbours by distance, so the
widest column would otherwise choose them alone, and the model inside iterative
imputation is yours to choose, and many models are sensitive to scale.

In [ ]:
# The plan is the thing under comparison. `partial` attaches it, and the fold
# arrives when the scorer runs it.
class ProportionalImputer:
    """Fill each missing label with one drawn from its column's training labels,
    in their training shares.

    scikit-learn has no such imputer, and `fill_missing` needs only `fit` and
    `transform`, so a short class is enough.
    """

    def __init__(self, random_state=None):
        self.random_state = random_state

    def fit(self, frame):
        self.shares = {c: frame[c].value_counts(normalize=True) for c in frame}
        return self

    def transform(self, frame):
        draws = np.random.default_rng(self.random_state)
        frame = frame.copy()
        for column, shares in self.shares.items():
            gaps = frame[column].isna()
            if gaps.any():
                frame.loc[gaps, column] = draws.choice(
                    shares.index, size=gaps.sum(), p=shares.to_numpy())
        return frame


# One imputer per fill. "own level" gives a missing label a category of its
# own instead of the common one.
fills = {"median": SimpleImputer(strategy="median"),
         "mean": SimpleImputer(strategy="mean"),
         "mode": SimpleImputer(strategy="most_frequent"),
         "zero": SimpleImputer(strategy="constant", fill_value=0),
         "own level": SimpleImputer(strategy="constant", fill_value="(missing)"),
         "proportional": ProportionalImputer(random_state=RANDOM_STATE),
         # Two that read the other numeric columns to fill each one.
         "KNN (k=5)": KNNImputer(n_neighbors=5),
         "iterative": IterativeImputer(random_state=RANDOM_STATE, max_iter=10)}

# Which numeric columns a mean would misrepresent, by the same skew measured
# in the distributions section, rather than by re-deriving it from dtype.
column_skew = deduped[numeric].skew()
skewed = column_skew[column_skew.abs() > 1].index.tolist()
symmetric = column_skew[column_skew.abs() <= 1].index.tolist()

plans = {
    "median / mode": [(numeric, fills["median"]), (categorical, fills["mode"])],
    "median / own level": [(numeric, fills["median"]),
                           (categorical, fills["own level"])],
    "median / proportional": [(numeric, fills["median"]),
                              (categorical, fills["proportional"])],
    "mean / mode": [(numeric, fills["mean"]), (categorical, fills["mode"])],
    "zero / mode": [(numeric, fills["zero"]), (categorical, fills["mode"])],
    "skew-aware": [(skewed, fills["median"]), (symmetric, fills["mean"]),
                   (categorical, fills["mode"])],
    # A deterministic fill where the constant MEANS something. Zero minutes of
    # sand training is a real amount of sand training; year zero is not a year.
    # So zero goes to the eleven duration columns and nowhere else, which is a
    # thing a plan can express and a single imputer cannot.
    "zero where none / median elsewhere": [
        (training_minutes, fills["zero"]),
        ([c for c in numeric if c not in training_minutes], fills["median"]),
        (categorical, fills["mode"])],
    # A label has no distance to measure, so these two fill numbers only.
    "KNN (k=5) / mode": [(numeric, fills["KNN (k=5)"]), (categorical, fills["mode"])],
    "iterative / mode": [(numeric, fills["iterative"]), (categorical, fills["mode"])],
}

plan_scores = {
    name: held_out_f1(partial(fill_missing, plan=plan),
                      deduped, deduped_splits, numeric, categorical)
    for name, plan in plans.items()}

plan_board = pd.DataFrame({
    "F1": {k: v.f1.mean() for k, v in plan_scores.items()},
    "F1 SEM": {k: sem(v.f1) for k, v in plan_scores.items()},
})
print(plan_board.round(4).to_string())

__Step 20:__ Compare every arm against the one the recipe ships, on the same twenty
splits.

In [ ]:
shipped = plan_scores["median / mode"]

# Paired: each arm minus the shipped arm on the SAME split, so the noise they
# share cancels and what is left is the difference between the arms.
paired = pd.DataFrame({
    name: {"dF1": (run.f1 - shipped.f1).mean(),
           "F1 SEM": sem(run.f1 - shipped.f1)}
    for name, run in plan_scores.items() if name != "median / mode"}).T
print(paired.round(4).to_string())

**`zero / mode` scores highest, +0.0035 ± 0.0017 above `median / mode`,
about two standard errors; every other plan is within one.** `zero where none /
median elsewhere` shows where that lead comes from. It writes 0 only into the
eleven duration columns, where zero minutes is a real amount, and gains
+0.0010 ± 0.0023, which is nothing. The rest of the lead is `Edition`, a year:
writing year 0 into its 21 empty cells marks those rows, so the fill works as a
missingness flag rather than as a value.

**The two imputers that read the other columns gain nothing.** KNN scores
-0.0004 ± 0.0019 and iterative -0.0011 ± 0.0018 against the training median:
more machinery, fitted twenty times, for no gain on this file. A more
elaborate fill is a candidate to measure, not an upgrade to assume.

**The recipe keeps `median / mode`.** The other label fills do not move it:
`median / own level` scores +0.0008 ± 0.0020 and `median / proportional`
0.0000 ± 0.0021. The mode adds no level to the feature space and draws nothing
at random, so it stays.

In [ ]:
# The fill plan this section chose.
chosen = "median / mode"
plan_scores[chosen].f1.mean()

__Step 21:__ Now the comparison that does move: fill, or drop?

In [ ]:
def drop_incomplete(train, test, plan):
    """Drop the training rows carrying any missing value, then fill what the rest need."""
    complete = ~train.isna().any(axis=1)
    return fill_missing(train[complete], test, plan)


keep_rows = plan_scores[chosen].f1
drop_rows = held_out_f1(partial(drop_incomplete, plan=plans[chosen]),
                        deduped, deduped_splits, numeric, categorical).f1
paired = keep_rows - drop_rows

print(f"impute the missing values  n={len(deduped):,}   F1 {keep_rows.mean():.4f}")
print(f"drop the rows    n={(~deduped.isna().any(axis=1)).sum():,}"
      f"   F1 {drop_rows.mean():.4f}")
print(f"paired difference {paired.mean():+.4f}"
      f" +/- {sem(paired):.4f}"
      f"   imputing wins on {(paired > 0).sum()} of {len(paired)} splits")

**Imputing keeps the 907 rows that dropping would delete (4,000 rows
against 3,093), and it is also the more accurate choice: 0.8182 against 0.8111,
a paired +0.0071 ± 0.0029, ahead on 16 of the 20 splits.**

What makes that comparison legitimate is that the drop arm drops rows from
the **training** side only, and both arms are scored on the same test rows. Score the
drop arm the tempting way (train *and* test on complete rows), and it is being
examined on an easier paper of its own choosing. 

In [ ]:
# Which plan do you carry forward, and why?
#
# median / mode. The nine plans sit within about two standard errors of each
# other, and the one ahead, zero / mode, gets its lead by writing year 0 into
# Edition's empty cells, which flags those rows instead of filling them. KNN
# and iterative imputation, the elaborate fills, gain nothing over the median.
#
# And the decision with clear evidence behind it: impute rather than drop.
# It keeps the 907 rows dropping would delete (4,000 against 3,093) and wins
# by +0.0071 +/- 0.0029 on identical held-out rows.

__Step 22:__ Build the spine the recipe ships. It is `deduped`, the frame the last
two comparisons ran on, with the booleans cast to a nullable dtype. The missing values stay open.

In [ ]:
# The spine is `deduped`, and nothing else. The chosen plan is NOT applied
# here: a median or a mode computed on this frame would be computed over the
# rows every later week holds out to score on, so the file would hand each week
# a value derived from its own test set. The plan is what later weeks apply,
# inside their own split, from their own training rows.
spine = deduped.copy()
# A nullable boolean, not `bool`: NaN counts as truthy, so astype(bool) would
# turn every missing value in these six columns into True.
for column in boolean_columns:
    spine[column] = spine[column].astype("boolean")

spine_splits = list(splits.split(spine, spine["Outcome"]))
print(f"spine: {spine.shape[0]:,} rows x {spine.shape[1]} columns,"
      f" win rate {spine['Outcome'].mean():.4f},"
      f" {spine.isna().sum().sum():,} missing values left open")

# <font color='#E8800A'>Outliers: compare five, then judge</font> <a class="anchor" id="outliers"></a>
[Back to TOC](#toc)

Same discipline as the last section: show the alternatives, decide on
measurement. The textbook answer loses, and it loses twice: once on structure,
once on bias.

The frame here is the spine as it will ship, missing values and all, so every
arm below fills them **inside its own split** from its own training rows, using
the plan the last section chose.

**The treatment runs first and the fill second**, and that order is a decision
rather than a default, because otherwise each step's statistic is computed over
the other step's output.

Fill first and the fence is drawn partly over cells the fill invented. A median
fill puts a spike of identical values at the centre of a column, which narrows
the box the fence is measured from. On the training half of the first split,
`Athlete score` carries **419 missing values in 3,200 rows**, and filling them all with one
number **halves its 1.5 × IQR fence, from a width of 240 to 120**. The rule then
looks stricter than the data warrants, on a column the fill itself made narrow.

Treating first is available only because `outside_fences` counts a missing value as inside.
Without that term a fence drawn before the fill would call all 907 rows carrying
a missing value outliers, which is the failure its docstring exists to prevent.

In [ ]:
def outside_fences(frame, columns, rule="iqr", k=1.5):
    """True for rows with a named column beyond its fences.

    `rule="iqr"` draws the fences k IQRs beyond the quartiles, `rule="zscore"`
    k standard deviations from the mean. A missing value counts as INSIDE: a
    comparison with a missing value is False both ways, so without the `isna()`
    term every row carrying one would be called an outlier.
    """
    values = frame[columns]
    if rule == "iqr":
        q1, q3 = values.quantile(0.25), values.quantile(0.75)
        low, high = q1 - k * (q3 - q1), q3 + k * (q3 - q1)
    else:
        mean, sd = values.mean(), values.std(ddof=0)
        low, high = mean - k * sd, mean + k * sd
    within = values.ge(low) & values.le(high)
    return ~(within | values.isna()).all(axis=1)

__Step 23:__ Apply the 1.5×IQR rule across every numeric column.

In [ ]:
# The detector defined above, with the 1.5 x IQR rule.
kept = ~outside_fences(spine, numeric, "iqr", k=1.5)
print(f"rows kept {kept.sum():,} of {len(spine):,}"
      f"  ({100 * (1 - kept.mean()):.2f}% removed)")
print(f"win rate  {spine['Outcome'].mean():.4f} ->"
      f" {spine.loc[kept, 'Outcome'].mean():.4f}")
print(f"win rate among the rows the rule DELETES:"
      f" {spine.loc[~kept, 'Outcome'].mean():.4f}")

<div class="alert alert-block alert-warning">

**The rule removes 67.33% of the dataset: 4,000 rows become 1,307.**
A diagnostic that calls two thirds of the data anomalous has not found anomalies;
it has failed. The previous section showed why: four columns have Q1 = Q3, so
their fences have zero width and every non-zero value is outside them. Stack
fourteen such tests with an *and* and almost nothing survives.

**And what it removes is not random. The win rate falls from 0.6040 to 0.3933,
and among the rows it deletes the win rate is 0.7063.** Winners train more, so
winners hold the extreme values, so a rule that deletes extreme values
**preferentially deletes the winners**. The loss is not 67% of the rows. It is
67% of the rows, weighted towards the ones carrying the signal.

A treatment that moves the class balance by twenty points has changed the
question, not cleaned the data.

</div>

__Step 24:__ Pin the z-score convention before anyone quotes a number.

In [ ]:
values = spine[numeric]
centred = values - values.mean()
scores_pop = centred / values.std(ddof=0)
scores_sample = centred / values.std(ddof=1)
print(f"|z| <= 3 with ddof=0 (numpy's default):"
      f" {(scores_pop.abs() <= 3).all(axis=1).sum():,} rows")
print(f"|z| <= 3 with ddof=1 (pandas' default):"
      f" {(scores_sample.abs() <= 3).all(axis=1).sum():,} rows")

<div class="alert alert-block alert-warning">

**Both conventions keep the same 2,690 rows here, and the
convention still has to be written down.** `numpy.std` divides by $n$ (`ddof=0`,
the population form); `pandas.Series.std` divides by $n-1$ (`ddof=1`, the sample
form). On 4,000 rows the two differ in the fourth decimal place of the fence,
and on this frame that missing value happens to be empty.

Happens to be. Nothing in either formula promises it, and on a smaller frame it
will not be: two students following the same instructions would then report
different answers, and neither would be wrong. **This notebook uses `ddof=0`
throughout**, stated here rather than left to whichever library you happened to
call. A rule with a tunable convention needs the convention written down with
it, and the argument for writing it down cannot depend on today's data being
the awkward case.

</div>

__Step 25:__ Name the five treatments with the exact limits their keys claim, then score
each on the shared splits and keep the treated frames for the table below.

In [ ]:
# Four arms, each one fold in and one fold out. Everything a treatment fits,
# it fits on `train`.
def keep_everything(train, test):
    """The control arm: the frame, untouched."""
    return train, test


def drop_outside(train, test, columns, rule, k):
    """Drop the TRAINING rows the fences call outliers; the test rows stay."""
    return train[~outside_fences(train, columns, rule, k)], test


def winsorize(train, test, columns, lower=0.01, upper=0.99):
    """Clip the named columns to the given TRAINING quantiles."""
    low, high = train[columns].quantile(lower), train[columns].quantile(upper)
    train, test = train.copy(), test.copy()
    train[columns] = train[columns].clip(low, high, axis=1)
    test[columns] = test[columns].clip(low, high, axis=1)
    return train, test


def log1p_of(train, test, columns):
    """log1p the named columns, clipping negatives to zero first. It fits nothing."""
    train, test = train.copy(), test.copy()
    train[columns] = np.log1p(train[columns].clip(lower=0))
    test[columns] = np.log1p(test[columns].clip(lower=0))
    return train, test


def treat_then_fill(train, test, plan, treatment):
    """Apply the treatment under test, then fill what is left inside the split.

    Treating first means a fence or a quantile is never fitted over cells the
    fill invented.
    """
    return fill_missing(*treatment(train, test), plan)


# log1p goes on the eleven training durations only: the three other numeric
# columns are a year, a count of attempts and a score, not long-tailed amounts.
treatments = {
    "keep everything": keep_everything,
    "drop IQR 1.5x": partial(drop_outside, columns=numeric, rule="iqr", k=1.5),
    "drop |z| > 3": partial(drop_outside, columns=numeric, rule="zscore", k=3),
    "winsorize 1/99": partial(winsorize, columns=numeric, lower=0.01, upper=0.99),
    "log1p": partial(log1p_of, columns=training_minutes),
}
treatment_scores = {
    name: held_out_f1(partial(treat_then_fill, plan=plans[chosen], treatment=fn),
                      spine, spine_splits, numeric, categorical).f1
    for name, fn in treatments.items()}

# What each treatment does to the rows it is allowed to touch: the TRAINING
# half of the first split, missing values still open, because that is what the
# treatment sees.
train_rows, test_rows = spine_splits[0]
first_train = spine.iloc[train_rows]
treated = {name: fn(first_train, first_train.iloc[:0])[0]
           for name, fn in treatments.items()}
baseline = treatment_scores["keep everything"]

__Step 26:__ Lay the five treatments side by side: rows kept, the class rate each
leaves, its F1, and the paired difference from keeping everything.

In [ ]:
per_treatment = {}
for name, frame in treated.items():
    scores = treatment_scores[name]
    difference = scores - baseline
    per_treatment[name] = {
        "rows kept": len(frame),
        "class rate": frame["Outcome"].mean(),
        "F1": scores.mean(),
        "vs keep": difference.mean(),
        "paired SEM": sem(difference),
        "beats keep on": f"{int((difference > 0).sum())} of {len(difference)}",
    }
outlier_board = pd.DataFrame.from_dict(per_treatment, orient="index")
print(outlier_board.round(4).to_string())

The same table as a chart: share of rows kept against win rate after
treatment.

In [ ]:
# Against the training half, because that is the only frame a treatment is
# allowed to touch -- dividing by the whole spine would understate every bar.
cost = (pd.DataFrame({"training rows kept": outlier_board["rows kept"] / len(first_train),
                      "win rate after": outlier_board["class rate"]})
        .rename_axis("treatment").reset_index()
        .melt(id_vars="treatment", var_name="measure", value_name="share"))

fig, ax = plt.subplots(figsize=FIGSIZE)
sns.barplot(cost, x="treatment", y="share", hue="measure",
            palette=[PLOT_BLUE, PLOT_ORANGE], ax=ax)
ax.axhline(first_train["Outcome"].mean(), color="black", ls="--", lw=1,
           label=f"win rate before treatment {first_train['Outcome'].mean():.3f}")
ax.set(xlabel="", ylabel="share", title="What each outlier treatment costs")
ax.tick_params(axis="x", labelrotation=15)
ax.legend()
plt.tight_layout()
plt.show()

**`log1p` on the eleven training durations wins, by a wide margin: 0.8506
against 0.8182 for keeping everything, a paired +0.0325 ± 0.0030, ahead on 20
of 20 splits.** The three other treatments sit within about a standard error of
keeping everything: `drop |z| > 3` +0.0025 ± 0.0016, `winsorize 1/99`
+0.0023 ± 0.0020 and `drop IQR 1.5×` +0.0022 ± 0.0045.

**The textbook rule also condemns itself on the other two columns.** `drop IQR
1.5×` deleted **2,145 of the 3,200 training rows**, keeping 1,055, and moved the
class rate from **0.6041 to 0.3943**, for a gain smaller than its own standard
error. In the chart, only `drop IQR 1.5×` moves the orange bar.

**`log1p` is the outlier treatment the recipe records.** It compresses the long
right tails instead of cutting them, and it fits nothing, so it cannot leak: the
spine keeps its units, and each later week applies it inside its own split.

Name the outlier decision the same way, as a value.

In [ ]:
# log1p won the board. It fits nothing, so it changes no row of the spine:
# the file keeps its units, and the log records the treatment for later weeks.
chosen_outlier_treatment = "log1p"

In [ ]:
# What did the outlier section decide, and what is carried forward?
#
# log1p on the eleven training durations: +0.0325 +/- 0.0030 against keeping
# everything, ahead on 20 of 20 splits. It compresses the long tails instead of
# cutting them and fits nothing, so the file keeps its units and each later
# week applies it inside its own split.
#
# Rejected: dropping rows by the 1.5*IQR rule. It removes 67.33% of the data
# and moves the win rate from 0.6040 to 0.3933, because the rows it deletes win
# at 0.7063, for +0.0022 +/- 0.0045, less than its own standard error.

__Step 27:__ Why the obvious comparison would have said the opposite.

In [ ]:
# The tempting order: fill and treat the WHOLE frame first, then split it.
# Both steps have then chosen their own test set, which is the error twice over
# -- and the fill here is the same whole-frame fill this notebook refused to
# bake into champions.csv, done in the one place it does real damage.
naive_frame, _ = fill_missing(spine, spine, plans[chosen])
naive = {}
for name in ["keep everything", "drop IQR 1.5x", "log1p"]:
    frame = treatments[name](naive_frame, naive_frame.iloc[:0])[0]
    matrix = design_matrix(frame, numeric, categorical)
    train_X, test_X, train_y, test_y = train_test_split(
        matrix, frame["Outcome"], test_size=0.2, stratify=frame["Outcome"],
        random_state=RANDOM_STATE)
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(train_X, train_y)
    naive[name] = f1_score(test_y, model.predict(test_X))

pd.DataFrame({"treat first, then split": naive,
              "split first, 20 splits":
                  {k: treatment_scores[k].mean() for k in naive}}).round(4)

<div class="alert alert-block alert-warning">

**The naive comparison, treating the whole dataset and then
cross-validating on it, does not only add noise. It can invert the answer.**

`drop IQR 1.5×` scores **0.6703** when the whole frame is treated and filled
before the split, and **0.8204** when each split is treated and filled on its
training rows only. Two very different stories about the same treatment, and
neither number is a typo.

The inversion runs the other way too, which is the more dangerous half. `keep
everything` reads **0.8300** treated first against **0.8182** treated inside
each split, and `log1p`
**0.8552** against **0.8506**: filling the whole frame before splitting flatters
both arms by about a point, because the fill has seen the rows they are
scored on. Neither number looks wrong on the page, which is why a leak is found
by checking the procedure rather than by reading the score.

The reason is that a treatment which removes rows changes its own test set. Drop
the hard rows and the remaining task is easier; or, as here, drop 68% of the
rows and the surviving class balance is so different that F1 collapses. Either
way the number is answering "how does this model do on the data this treatment
left behind?", which is not the question. **Only identical, untouched test rows
make five numbers comparable.**

</div>

# <font color='#E8800A'>Does scaling change anything here?</font> <a class="anchor" id="scaling"></a>
[Back to TOC](#toc)

Every comparison above fits logistic regression on unscaled columns, and
the solver stops at its iteration cap on them. Week 4 chooses the scaler together
with the encoder; this is a first look at what a scaler changes.

__Step 28:__ Score four scalers on the recipe's frame.

In [ ]:
# The same scorer, handed a `scaler`: every row below uses the same model, the
# recipe's frame (log1p on the eleven durations, median / mode inside each
# split) and the same twenty splits.
scaler_candidates = {
    "unscaled": None,
    "standard": StandardScaler(),
    "min-max": MinMaxScaler(),
    "robust": RobustScaler(),
}
recipe_frame = partial(treat_then_fill, plan=plans[chosen],
                       treatment=treatments[chosen_outlier_treatment])
scaler_runs = {
    name: held_out_f1(recipe_frame, spine, spine_splits, numeric, categorical,
                      scaler=scaler)
    for name, scaler in scaler_candidates.items()
}
scaler_board = pd.DataFrame({
    "F1": {name: run.f1.mean() for name, run in scaler_runs.items()},
    "SEM": {name: sem(run.f1) for name, run in scaler_runs.items()},
    "iterations": {name: run.iterations.mean()
                   for name, run in scaler_runs.items()},
})
print(scaler_board.round({"F1": 4, "SEM": 4, "iterations": 0}).to_string())

**The four scalers sit within one standard error of each other on F1**:
0.8523 standard, 0.8522 robust, 0.8517 min-max and 0.8506 unscaled. **What
scaling changes is the solver.** Standard scaling converges in about 22
iterations, robust in 52 and min-max in 90, while the unscaled fit stops at the
1,000-iteration cap. The score does not choose a scaler here, so this week's log
records none, and Week 4 benchmarks the scaler together with the encoder.

# <font color='#E8800A'>The recipe, the order, and the log</font> <a class="anchor" id="recipe"></a>
[Back to TOC](#toc)

Six questions were opened at the end of the exploration and each
has been answered by running its alternatives through the function that would
carry out the winner. What is left is to call those functions once, in
order, on the file as it arrived, and to record why each one is there.

<div class="alert alert-block alert-info">

**The recipe, what each step chose, and the one step it records rather
than applies.**

| # | operation | what won, and what it beat | what it changes |
|---|---|---|---|
| 1 | text repair, six boolean columns | not a comparison: `'FASE'` is a misspelling, not a treatment | 23 cells |
| 2 | invalid values | **blank the cell**, against clip-to-zero, drop-the-row and keep-and-flag | +513 missing |
| 3 | duplicate athletes | **keep the earliest `RecordID`**, chosen for reproducibility after the score refused to separate the rules | −280 rows |
| 4 | missing values | **median / mode**, against eight other plans | 1,014 cells left open |
| | **result** | | 4,280 → 4,000 rows, written to `champions.csv` |

**Row 4 is recorded, not applied.** A fill reads other rows to decide a cell's
value, so applying one here would compute it over rows that later become
someone's test set. The plan is written into the log instead, and each later
training split computes its own values from it.

**One more decision is recorded the same way**, because it belongs to a model
rather than to the file: the outlier treatment, **`log1p` on the eleven duration
columns**, which beat the four others by +0.0325 on 20 of 20 splits. The log
below records every step and is written to `logs/champions_cleaning_log.json`.

**Each step carries its decision twice**: once as the sentence you just read,
and once as `carries`, a small structure a later week can act on. A step
described only in English can be read and not applied, because the plan has to
be worked out again from the sentence, and a plan worked out again is a new
decision wearing an old one's name. So what carries forward is the dataset, a
written account of how it was made, and the plan itself.

</div>

__Step 29:__ Run the recipe and record every step as you go.

In [ ]:
log = CleaningLog("champions")
frame = raw.copy()
frame[boolean_columns] = raw[boolean_columns].astype("object").replace(text_fixes)
log.record("6 boolean columns", "'FASE', 'FALSE' and 'TRUE' to real booleans",
           "one misspelling and one casing convention would otherwise leave "
           "'Mental preparation' a three-category column",
           int((raw["Mental preparation"] == "FASE").sum()),
           carries={"cast": {c: "boolean" for c in boolean_columns},
                    "replace": text_fixes})

nulls_before = frame.isna().sum().sum()
frame = fix_invalid(frame, invalid_cols, invalid_errors, np.nan)
log.record("Physiotherapy, Athlete score, Age group",
           f"invalid values -> NaN ('{chosen_invalid_fix}')",
           "a duration and a 0-140 score cannot be negative, and '0' is not one "
           "of Age group's three bands; 499 of the 505 negatives are exactly "
           "-30, concentrated in 3 of 7 competitions, an administrative "
           "placeholder rather than missing at random; blanking tied with clipping "
           "and with flagging, and beat dropping the row by +0.0109 F1",
           int(frame.isna().sum().sum() - nulls_before),
           carries={"treatment": "blank", "becomes": None,
                    "columns": list(invalid_cols)})

rows_before = len(frame)
frame = deduplicate(frame, "Athlete Id", tiebreaks[chosen_tiebreak])
log.record("Athlete Id", f"keep each athlete's {chosen_tiebreak}",
           "there are no duplicate ROWS; the repetition is one athlete across "
           "several competitions, and an athlete on both sides of a split hands "
           "the model the answer; no tie-break scored better than any other, so "
           "this one is recorded for reproducibility rather than for accuracy",
           rows_before - len(frame),
           carries={"key": "Athlete Id", "keep": chosen_tiebreak})

# The fourth decision changes no cell, which is exactly why it is logged: a
# reader has to be able to see that leaving the missing values open was chosen.
log.record("26 columns", f"missing values LEFT OPEN; the '{chosen}' plan goes to later weeks",
           "a median or a mode computed here would be computed over the rows "
           "each later week holds out to score on, so the file would carry a "
           "value derived from its own test set; the plan won the comparison "
           "above and each week applies it inside its own split",
           int(frame.isna().sum().sum()),
           # The one step nothing here applies, so the one whose `carries`
           # does the most work: a later week reads the plan and fits it on
           # its own training rows.
           carries={"applied": False, "fill": {"numeric": "median",
                                               "categorical": "mode"},
                    "numeric": numeric, "categorical": categorical})

# The outlier treatment belongs to a model rather than to the file, so it is
# recorded and not applied either. `outliers: "keep"` tells a later week to
# clip or drop nothing beyond the log1p.
log.record("11 training-volume columns",
           "outlier treatment: log1p, and log1p on nothing else",
           "five outlier treatments were compared on 20 paired splits of identical "
           "held-out rows; log1p won, F1 0.8506 against 0.8182 for keeping every "
           "row, a paired +0.0325 +/- 0.0030, better on 20 of 20; dropping rows by "
           "the 1.5*IQR rule removes 67.33% of the file for +0.0022 +/- 0.0045; "
           "log1p fits nothing, so each later week applies it inside its own split",
           len(training_minutes),
           carries={"applied": False, "transform": "log1p",
                    "columns": training_minutes, "outliers": "keep",
                    "rule_tested": "log1p against IQR, z-score and winsorizing"})

record = pd.DataFrame(log.records())
print(record.drop(columns="reason").to_string(index=False))
print(f"\n{len(raw):,} rows in -> {len(frame):,} rows out")

# The log is written, not just printed, so a later week can open the four
# decisions that produced the file it is about to model on. JSON, not CSV,
# because each step carries a machine-readable form of its decision beside the
# prose, and a CSV would flatten that back into text.
log.to_json("../../logs/champions_cleaning_log.json")
print(f"log written: {len(log)} decisions,"
      f" {sum(step.carries is not None for step in log.steps)} carrying a plan")

**23 cells, then 513 more missing values, then 280 rows out, and 1,014 missing values
left open.** The first three rows account for every difference between the file
that arrived and the file that leaves. The fourth accounts for a difference that
was deliberately NOT made, which a diff cannot show you and a log can. The
`reason` column is dropped only to fit the page: it is the column that makes
this a log rather than a diff.

__Step 30:__ The seam: does the recipe reproduce the spine?

In [ ]:
cleaned = frame.copy()
for column in boolean_columns:
    cleaned[column] = cleaned[column].astype("boolean")

# A CSV stores text, not dtypes, so the six boolean columns are cast before
# writing and whoever reads the file has to ask for the same dtype. With missing values in
# them `bool` is the wrong cast, because NaN is truthy.
cleaned.to_csv("../../data/interim/champions.csv")
print(f"written: {cleaned.shape[0]:,} rows x {cleaned.shape[1]} columns,"
      f" {cleaned.isna().sum().sum():,} missing values")

<div class="alert alert-block alert-success">

**4,280 rows became 4,000**, with 1,014 cells left open on
purpose. The cleaned file is not handed down from anywhere: it is the output of
the four log rows above, and you have just produced it.

</div>

<div class="alert alert-block alert-warning">

**Why the file ships with its missing values open.**

The tempting argument for filling here is that this is dataset **preparation**,
happening before any train/test split exists, so a median over the whole frame
cannot leak into a score that does not exist yet. It is wrong, and the reason it
is wrong is that the file outlives the argument. Every later week splits THIS
file, so a median computed over all 4,000 rows is a median computed over each of
those weeks' test rows, and the leak arrives later even though the code ran
earlier.

That is why the plan the comparison chose is a **recipe rather than a value**.
1,014 cells stay empty, each later week fills them from its own training rows,
and no week is scored on a number its own test set helped compute. The cost is
one line per notebook. The alternative is a course that teaches the discipline
in one week and breaks it in the file it hands out.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

What this session established:

1. **A raw file is not a dataset.** The clean frame the rest of the course opens
   is produced here, by decisions you can name; it is not handed to you.
2. **An invalid value is not missing data until you decide it is.** A zero
   that cannot physically be zero, a placeholder value, an impossible category: find
   those before you count what is missing, or the count is wrong. What they
   should become is a separate question, and this file answered it by
   measurement rather than by definition.
3. **Compare strategies, then select one.** Missing value filling and outlier handling
   were each run several ways on the same frame and judged against each other,
   which is the only way to know the one you kept was worth keeping. Here the
   elaborate fills, KNN and iterative imputation, gained nothing over the median.
4. **A summary hides things, and a boxplot is a summary.** The shape of a
   distribution decides which technique applies to it, so look at the shape
   before reaching for the technique.
5. **An outlier treatment can reshape rather than remove.** `log1p` beat every
   rule that drops or clips rows, and it keeps every row the file has.
6. **The recipe is part of the result.** What changed, what stayed open and why
   are artifacts handed over with the data, not facts a later reader should have
   to reconstruct from the finished file.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Little, R. J. A. & Rubin, D. B. (2019). *Statistical Analysis with Missing Data*, 3rd ed. Wiley: missing-data mechanisms, and why "not missing at random" changes what an imputation means.
- van Buuren, S. (2018). *Flexible Imputation of Missing Data*, 2nd ed. CRC Press: the reference behind `IterativeImputer`; § 1.3 on why complete-case analysis is rarely the safe default.
- Tukey, J. W. (1977). *Exploratory Data Analysis*. Addison-Wesley: the source of the 1.5×IQR rule, including its stated assumptions.
- Aggarwal, C. C. (2017). *Outlier Analysis*, 2nd ed. Springer: § 1.2 on why an outlier is defined relative to a model, not to a formula.
- scikit-learn User Guide, [Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html) · [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) · [`KNNImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html) · [`IterativeImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html).